# Limpieza de los Datos de la Tabla bronce.oficiales_credito para Cargalos en la Capa Plata

Proposito del script:  
- Verificar columna por columna los tipos de datos para encontrar inconsistencias en los datos.  
- Limpiar y estandarizar, columna por columna los datos.  
- Exportar la nueva tabla como un archivo con nombre "semi_limpio_oficiales_credito.parquet".

# Estableciendo Conexion

In [1]:
# Importando las librerias y creando la conexión 
import pandas as pd 
from funciones import limpiar_texto,formato_genero
from conexiones_y_rutas import obtener_engine,obtener_ruta_archivo
engine = obtener_engine()
df_oficiales_credito = pd.read_sql(
    "SELECT * FROM bronce.oficiales_credito",
    con=engine
)

df_oficiales_tra = df_oficiales_credito.copy()

# Archivos de Ayuda

In [2]:
# Tabla limpia de sucursales, para validar, surcursal_id
df_sucursal = pd.read_parquet(obtener_ruta_archivo("archivos_limpios","limpio_sucursales.parquet"))
df_sucursal_tra = df_sucursal.copy()
df_sucursal_tra.head()

,sucursal_id,codigo_sucursal,nombre_sucursal,tipo_sucursal,ciudad,departamento,region,zona,fecha_apertura,estado_sucursal
0,1,SUC0001,Oficina Principal Lima,Oficina Principal,San Juan de Lurigancho,Lima,Lima y Callao,Urbano,2005-07-24,Activa
1,2,SUC0002,Agencia Santiago de Surco 1,Agencia,Santiago de Surco,Lima,Lima y Callao,Urbano,2010-01-03,Activa
2,3,SUC0003,Agencia Miraflores 2,Punto de Atención,Miraflores,Lima,Lima y Callao,Urbano,2007-04-20,Activa
3,4,SUC0004,Agencia los Olivos 3,Agencia,Los Olivos,Lima,Lima y Callao,Urbano,2010-02-19,Activa
4,5,SUC0005,Agencia Lima 4,Agencia,Lima,Lima,Lima y Callao,Urbano,2007-02-07,Activa


# Resumen de las Columnas 

- **oficial_id**: Identificador unico de cada oficial.  
- **sucursal_id**: Identificador de la sucursal asociada al oficial de credito.   
- **nombres**:  Nombres del oficial.  
- **apellido_paterno**: Apellido parterno del oficial.  
- **apellido_materno**:  Apellido materno del oficial.  
- **genero**: Genero del oficial (ejem: Masculino o Femenino).  
- **cargo**: Cargo del oficial (ejem: Analista de Creditos o Analista Senior de Credito).  
- **fecha_ingreso**: Fecha de ingreso del oficial de credito.  
- **estado**: Estado actual del oficial de credito (ejem: Activo o Inactivo)

# Verificacion de la Calidad y Limpieza de los Datos

In [3]:
df_oficiales_tra.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   oficial_id        62 non-null     int64 
 1   sucursal_id       62 non-null     int64 
 2   nombres           62 non-null     object
 3   apellido_paterno  62 non-null     object
 4   apellido_materno  62 non-null     object
 5   genero            62 non-null     object
 6   cargo             59 non-null     object
 7   fecha_ingreso     62 non-null     object
 8   estado            62 non-null     object
dtypes: int64(2), object(7)
memory usage: 4.5+ KB


In [4]:
df_oficiales_tra.head()

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado
0,1,12,Alejandra,Herrera,Rivera,F,Analista de Créditos,2017/12/20,Activo
1,2,24,Víctor,Rivera,Cusi,M,Analista de Créditos,2015/06/29,Activo
2,3,8,Héctor,Vargas,López,M,None,20-09-2010,Activo
3,4,7,Fernando,Laime,Vargas,M,Analista Senior de Créditos,2021-03-13,Activo
4,5,8,Patricia,Ortiz,Ramírez,F,None,2022/08/04,Activo


In [5]:
# Verifica si existen registro duplicados 
df_oficiales_tra[df_oficiales_tra.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado


In [6]:
# Elmina registros duplicados
df_oficiales_tra.drop_duplicates(inplace=True)
df_oficiales_tra.reset_index(drop=True,inplace=True)

## oficial_id

In [7]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_oficiales_tra[df_oficiales_tra.oficial_id <= 0]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado


In [8]:
# Verifica si existen ids duplicados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra[df_oficiales_tra.oficial_id.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado
16,17,11,Ricardo,González,Martínez,M,Analista Senior de Créditos,2011/08/04,Activo
52,53,7,Héctor,Ramos,Cruz,M,Analista de Créditos,2013/04/22,Activo
60,53,7,Héctor,Ramos,Cruz,M,Analista de Créditos,2013-04-22,Activo
61,17,11,Ricardo,González,Martínez,M,Analista Senior de Créditos,2011-08-04,Activo


## sucursal_id

In [9]:
# Verifica si existen ids negativos o 0 
# Resultados Esperados: Tabla Vacia
df_oficiales_tra[df_oficiales_tra.sucursal_id <= 0]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado


In [10]:
# Verifica que efectivamente los id de sucursales existan en la tabla sucursales 
# Resultados Esperados: both: 62, left_only: 0, right_only: 0
verificando_ids = df_oficiales_tra.merge(
    right=df_sucursal_tra,
    on='sucursal_id',
    how='left',
    indicator=True)
verificando_ids._merge.value_counts()

_merge
both          62
left_only      0
right_only     0
Name: count, dtype: int64

## nombres

In [11]:
# Verifica si existen nombres con formatos inadecuados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra.nombres[df_oficiales_tra.nombres != df_oficiales_tra.nombres.str.strip().str.title()]

5       Fabiola
8        Ángela
10        jorge
21        césar
26     RoBeRtO 
51       HéCtOr
52     Héctor  
60     Héctor  
Name: nombres, dtype: object

In [12]:
# Aplica el formato de texto adecuado para los nombre y verifica si existen diferencias
df_oficiales_tra["nombres"] = df_oficiales_tra.nombres.apply(limpiar_texto)
df_oficiales_tra.nombres[df_oficiales_tra.nombres != df_oficiales_tra.nombres.str.strip().str.title()]

Series([], Name: nombres, dtype: object)

## apellido_paterno

In [13]:
# Verifica si existen apellidos con formatos inadecuados
# Resultados Esperados: Tabla Vacia 
df_oficiales_tra.apellido_paterno[df_oficiales_tra.apellido_paterno 
                                != df_oficiales_tra.apellido_paterno.str.strip().str.title()]

11       Torres
15      Choque 
27    MARTÍNEZ 
35      CasTro 
59      Romero 
Name: apellido_paterno, dtype: object

In [14]:
# Aplica el formato de texto adecuado para apellido_paterno y verifica si existen diferencias
df_oficiales_tra["apellido_paterno"] = df_oficiales_tra.apellido_paterno.apply(limpiar_texto)
df_oficiales_tra.apellido_paterno[df_oficiales_tra.apellido_paterno 
                                != df_oficiales_tra.apellido_paterno.str.strip().str.title()]

Series([], Name: apellido_paterno, dtype: object)

## apellido_materno

In [15]:
# Verifica si existen apellidos con formatos inadecuados
# Resultados Esperados: Tabla Vacía 
df_oficiales_tra.apellido_materno[df_oficiales_tra.apellido_materno 
                                != df_oficiales_tra.apellido_materno.str.strip().str.title()]

5        Herrera
21       SÁNCHEZ
24        chávez
29      García  
40     Gutiérrez
44     Martínez 
Name: apellido_materno, dtype: object

In [16]:
# Aplica el formato de texto adecuado para apellido_materno y verifica si existen diferencias
df_oficiales_tra["apellido_materno"] = df_oficiales_tra.apellido_materno.apply(limpiar_texto)
df_oficiales_tra.apellido_materno[df_oficiales_tra.apellido_materno 
                                != df_oficiales_tra.apellido_materno.str.strip().str.title()]

Series([], Name: apellido_materno, dtype: object)

## genero 

In [17]:
# Resultados Esperados: "FEMENINO", "MASCULINO", "n/a"
df_oficiales_tra.genero.unique()

array(['F', 'M', 'fem', 'F ', 'Masculino', 'FEMENINO', 'f'], dtype=object)

In [18]:
# Estandarizamos los resultados para los generos 
df_oficiales_tra["genero"] = df_oficiales_tra.genero.apply(formato_genero)
df_oficiales_tra.genero.unique()

array(['Femenino', 'Masculino'], dtype=object)

## cargo

In [19]:
# Resultados Esperados: 'Analista de Créditos', 'n/a', 'Analista Senior de Créditos', 'Oficial de Créditos', 'Oficial Senior', 'Supervisor de Créditos'
df_oficiales_tra.cargo.unique()

array(['Analista de Créditos', None, 'Analista Senior de Créditos',
       'Oficial de Créditos', 'Oficial Senior', 'OfIcIaL De cRéDiToS ',
       ' Analista de Créditos ', 'Supervisor de Créditos',
       'SUPERVISOR DE CRÉDITOS', 'ANALISTA SENIOR DE CRÉDITOS',
       'AnAlIsTa sEnIoR De cRéDiToS ', 'OFICIAL DE CRÉDITOS',
       ' Analista de créditos'], dtype=object)

In [20]:
# Estandarizamos los resultados para el cargo
# Resultados Esperados: 'Analista de Créditos', 'n/a', 'Analista Senior de Créditos', 'Oficial de Créditos', 'Oficial Senior', 'Supervisor de Créditos'
df_oficiales_tra["cargo"] = df_oficiales_tra.cargo.apply(limpiar_texto)
df_oficiales_tra.cargo.unique()

array(['Analista de Créditos', 'n/a', 'Analista Senior de Créditos',
       'Oficial de Créditos', 'Oficial Senior', 'Supervisor de Créditos'],
      dtype=object)

## fecha_ingreso

In [21]:
# Muestra las fechas que generan errores al trasformar a datetime 
# Resultados Esperados: Tabla Vacia
fecha_ingreso_error = pd.to_datetime(df_oficiales_tra.fecha_ingreso, errors='coerce')
df_oficiales_tra.fecha_ingreso[fecha_ingreso_error.isna()]

2     20-09-2010
3     2021-03-13
6     21-06-2019
7     27-07-2012
8     2019-10-02
10    2021-11-23
12    27-10-2012
15    2014-12-10
20    2019-06-19
23    2020-03-16
24    25/04/2011
25    29-11-2021
26    2015-04-11
27    2023-02-03
28    17/11/2018
30    21-12-2012
31    2022-03-09
34    2015-08-20
35    30-01-2010
37    24/05/2022
38    2015-08-07
39    21-12-2013
40    2012-06-04
41    2016-11-05
42    2016-04-01
45    2022-12-07
49    2022-04-16
50    2019-12-02
51    2024-03-13
53    16-02-2024
54    2010-02-15
55    14/08/2021
60    2013-04-22
61    2011-08-04
Name: fecha_ingreso, dtype: object

In [22]:
# Transforma las fechas a datetime, usando format='mixed'
df_oficiales_tra["fecha_ingreso"] = pd.to_datetime(
    df_oficiales_tra.fecha_ingreso,
    errors='coerce',
    format='mixed',
    dayfirst=True
)
df_oficiales_tra.fecha_ingreso[fecha_ingreso_error.isna()]

2    2010-09-20
3    2021-03-13
6    2019-06-21
7    2012-07-27
8    2019-10-02
10   2021-11-23
12   2012-10-27
15   2014-12-10
20   2019-06-19
23   2020-03-16
24   2011-04-25
25   2021-11-29
26   2015-04-11
27   2023-02-03
28   2018-11-17
30   2012-12-21
31   2022-03-09
34   2015-08-20
35   2010-01-30
37   2022-05-24
38   2015-08-07
39   2013-12-21
40   2012-06-04
41   2016-11-05
42   2016-04-01
45   2022-12-07
49   2022-04-16
50   2019-12-02
51   2024-03-13
53   2024-02-16
54   2010-02-15
55   2021-08-14
60   2013-04-22
61   2011-08-04
Name: fecha_ingreso, dtype: datetime64[ns]

**Nota** No estoy muy seguro de esta fecha de ingreso, porque:  
- Puede que sea la fecha desde que ingreso a la empresa en X fecha pero luego se cambiara a una nueva sucursal entonces puede aparecer que ingreso a una sucursal antes de que esta aperturara.  
- Tambien puede que sea la fecha de ingreso para esa sucursal asociada.

In [23]:
# Columnas a utilizar
df_oficiales_revi = df_oficiales_tra[['oficial_id','sucursal_id','fecha_ingreso']].copy()
df_sucursal_revi = df_sucursal_tra[['sucursal_id','fecha_apertura']].copy()
# LEFT JOIN
df_merge_fechas = df_oficiales_revi.merge(
    right = df_sucursal_revi,
    how='left',
    on='sucursal_id'
)
# Selecciona los registros incorrectos
# Resultados Esperados: Tabla Vacia 
df_merge_fechas[df_merge_fechas.fecha_ingreso < df_merge_fechas.fecha_apertura]

,oficial_id,sucursal_id,fecha_ingreso,fecha_apertura


## estado

In [24]:
# Resultado Esperado: 'Activo', 'Inactivo', 'n/a'
df_oficiales_tra.estado.unique()

array(['Activo', 'Activo ', 'Inactivo', 'activo', 'inactivo', 'activo '],
      dtype=object)

In [25]:
df_oficiales_tra["estado"] = df_oficiales_tra.estado.apply(limpiar_texto)
df_oficiales_tra.estado.unique()

array(['Activo', 'Inactivo'], dtype=object)

# Limpiando Duplicados Luego de Limpieza

In [26]:
# Verifica si existen duplicados 
df_oficiales_tra[df_oficiales_tra.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado
16,17,11,Ricardo,González,Martínez,Masculino,Analista Senior de Créditos,2011-08-04,Activo
52,53,7,Héctor,Ramos,Cruz,Masculino,Analista de Créditos,2013-04-22,Activo
60,53,7,Héctor,Ramos,Cruz,Masculino,Analista de Créditos,2013-04-22,Activo
61,17,11,Ricardo,González,Martínez,Masculino,Analista Senior de Créditos,2011-08-04,Activo


In [27]:
# Elimina duplicados
df_oficiales_tra.drop_duplicates(inplace=True)

## oficial_id V2

In [28]:
# Muestra identificadores de oficiales duplicados
df_oficiales_tra[df_oficiales_tra.oficial_id.duplicated(keep=False)]

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado


# Exportando la Tabla Limpia

In [29]:
df_oficiales_tra.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   oficial_id        60 non-null     int64         
 1   sucursal_id       60 non-null     int64         
 2   nombres           60 non-null     object        
 3   apellido_paterno  60 non-null     object        
 4   apellido_materno  60 non-null     object        
 5   genero            60 non-null     object        
 6   cargo             60 non-null     object        
 7   fecha_ingreso     60 non-null     datetime64[ns]
 8   estado            60 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(6)
memory usage: 4.7+ KB


In [30]:
df_oficiales_tra.head()

,oficial_id,sucursal_id,nombres,apellido_paterno,apellido_materno,genero,cargo,fecha_ingreso,estado
0,1,12,Alejandra,Herrera,Rivera,Femenino,Analista de Créditos,2017-12-20,Activo
1,2,24,Víctor,Rivera,Cusi,Masculino,Analista de Créditos,2015-06-29,Activo
2,3,8,Héctor,Vargas,López,Masculino,n/a,2010-09-20,Activo
3,4,7,Fernando,Laime,Vargas,Masculino,Analista Senior de Créditos,2021-03-13,Activo
4,5,8,Patricia,Ortiz,Ramírez,Femenino,n/a,2022-08-04,Activo


In [31]:
df_oficiales_tra.to_parquet(
    obtener_ruta_archivo("archivos_semi_limpios","semi_limpio_oficiales_credito.parquet"),
    index=False
)